# Build the Logan bioplastics-metadata variants

This corrected experiment defines **metadata** as plastic-degradation and biochemical metadata from PlasticDB, PAZy, and PETadex-derived classifications. ENA fields are retained only to document study context and provenance.

The sequence payload is identical in both databases. The enriched database adds verified plastic-study context, a protein reference library, plastic and enzyme annotations, ProtParam biochemical properties, DIAMOND translated-homology hits, and provenance.

Six previously sampled Logan runs serve as background controls. Three Logan runs from the verified microplastic study PRJDB39034 provide plastic-related study context. Study context is not treated as proof of degradation activity.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from bioplastics_benchmark_core import ROOT, prepare_bioplastics_benchmark
root = ROOT

In [2]:
(downloads, studies, references, annotations, hits, controls,
 homology, builds, validation, summary) = prepare_bioplastics_benchmark(root)
display(summary)
display(studies[['run_accession','study_accession','study_title','context_class','library_source','library_strategy']])

,logan_accessions,plastic_context_accessions,sequence_records,total_bases,reference_sequences,reference_annotations,candidate_similarity_rows
0,9,3,121326,18020607,323,534,28


,run_accession,study_accession,study_title,context_class,library_source,library_strategy
0,DRR001152,PRJDA68269,Canis lupus familiaris Transcriptome project,background_control,TRANSCRIPTOMIC,RNA-Seq
1,DRR000584,PRJDA50789,Botryococcus braunii BOT22 transcriptome project,background_control,TRANSCRIPTOMIC,EST
2,DRR001026,PRJDA67119,Oryza sativa Japonica Group strain:Nipponbare ...,background_control,TRANSCRIPTOMIC,RNA-Seq
3,DRR001104,PRJDA63487,Pinctada fucata utokyo_mb transcriptome project,background_control,TRANSCRIPTOMIC,EST
4,DRR000185,PRJDA42805,Whole SNPs analysis of ciprofloxacin resistanc...,background_control,GENOMIC,WGS
5,DRR001199,PRJDB2651,454 libraries of PCR products for T cell recep...,background_control,TRANSCRIPTOMIC,AMPLICON
6,DRR821254,PRJDB39034,Multi-omics reveals the response of anaerobic ...,plastic_context,GENOMIC,OTHER
7,DRR821257,PRJDB39034,Multi-omics reveals the response of anaerobic ...,plastic_context,GENOMIC,OTHER
8,DRR821260,PRJDB39034,Multi-omics reveals the response of anaerobic ...,plastic_context,GENOMIC,OTHER


## Reference-library provenance and biochemical metadata

Raw PlasticDB sequences are the core reference set. PAZy UniProt accessions are resolved and cached when available. Canonical protein identity is based only on sequence SHA-256, so exact cross-source duplicates are stored once while source-specific annotation rows are preserved. PETadex study logic supplies versioned polymer-family normalization, enzyme-family keyword rules, and ProtParam fields. The broad polymer class means bio-based and/or biodegradable; commercial formulations such as Impranil and Ecovio-FT are reported separately, and natural rubber is classified as a natural biopolymer rather than a plastic. A generated audit table covers every distinct PlasticDB plastic label.

In [3]:
display(references.describe(include='all').transpose())
display(annotations.groupby(['source','plastic_class']).size().rename('annotation_rows').reset_index())
display(annotations.groupby('enzyme_family').size().sort_values(ascending=False).rename('annotation_rows').head(15))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
reference_id,323,323,ref_00dc2f0dc3b242a2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sequence,323,323,MVKRIGFAAAIGLVILAVVAPGSAPAAESPYQRGPDPTRESVAASR...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sequence_sha256,323,323,00dc2f0dc3b242a2c2aab86266821d3f6e6b23272e126a...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
protein_length,323.0,NaN,NaN,NaN,360.944272,126.443135,108.0,279.5,311.0,429.5,1047.0
molecular_weight,323.0,NaN,NaN,NaN,38450.29183,13542.436749,12615.3345,29946.4991,33655.8194,45579.5466,117020.6819
isoelectric_point,323.0,NaN,NaN,NaN,7.18257,1.710394,4.17911,5.693499,6.947339,8.752889,11.409115
instability_index,323.0,NaN,NaN,NaN,35.006142,8.480957,12.82905,29.1894,34.837021,39.939677,99.817742
gravy,323.0,NaN,NaN,NaN,-0.143732,0.171,-1.682581,-0.242617,-0.149449,-0.029774,0.239352
aromaticity,323.0,NaN,NaN,NaN,0.08635,0.018205,0.003226,0.077217,0.086162,0.097155,0.166667


,source,plastic_class,annotation_rows
0,PAZy,bioplastic_or_biodegradable_polymer,5
1,PAZy,other_plastic,8
2,PlasticDB,bioplastic_or_biodegradable_polymer,283
3,PlasticDB,commercial_biodegradable_formulation,2
4,PlasticDB,natural_biopolymer,4
5,PlasticDB,other_plastic,232


enzyme_family
PETase                145
Depolymerase          135
Other enzyme          102
Lipase/esterase        77
Cutinase               60
Protease                8
Alkane hydroxylase      3
Laccase                 2
Oxidase                 2
Name: annotation_rows, dtype: int64

## Homology and validation controls

DIAMOND `blastx` compares Logan nucleotide contigs against the protein library. A candidate similarity requires E-value <= 1e-5, identity >= 30%, aligned length >= 50 amino acids, and reference coverage >= 50%. This is a permissive retrieval filter, not a plastic-function classifier. Synthetic codon-encoded reference proteins and shuffled artificial decoys test pipeline plumbing only; they do not estimate biological sensitivity or specificity.

In [4]:
print(controls)
print(homology)
display(hits.head(20))

{'positive_controls': 25, 'expected_reference_recovered': 25, 'exact_sequence_pipeline_recovery_rate': 1.0, 'negative_controls': 25, 'artificial_decoys_with_candidate_hit': 0, 'artificial_decoy_hit_rate': 0.0, 'control_scope': 'Pipeline plumbing check using exact codon-encoded references and shuffled artificial decoys; not biological sensitivity or specificity.', 'thresholds': {'evalue_max': 1e-05, 'identity_min_pct': 30.0, 'aligned_aa_min': 50, 'subject_coverage_min': 0.5}}
{'diamond_version': 'diamond version 2.1.11', 'query_sequences': 121326, 'raw_alignment_rows': 51, 'candidate_alignment_rows': 28, 'sequences_with_candidate_similarity': 18, 'runtime_seconds': 3.624155965999762, 'query_fasta_sha256': '105cc59b55a97e1999e5dbcdb7365aac71dd29ed7c0486decf77a67af9e563e2', 'reference_fasta_sha256': 'e59e7ca9a3ef1cb280b6e5af933c1356fc2771cc0c309b3f6e936a245c6c3a3c', 'raw_output_sha256': '9cb81a20284133e0e632768264a9f258a43e1201def1082ae951ae98fa76c4d3', 'thresholds': {'evalue_max': 1e-05,

,sequence_id,reference_id,identity_pct,aligned_aa,query_length_nt,subject_length_aa,subject_start,subject_end,evalue,bitscore,subject_coverage,hit_rank
38,DRR000185_1066,ref_a1e3671a8fb81e5e,30.2,434,1418,467,48,466,1.010000e-57,192.0,0.897216,1
39,DRR000185_1467,ref_067339e228f8479f,59.4,234,689,402,17,246,1.490000e-100,292.0,0.572139,1
40,DRR000185_1467,ref_1f9b512005633c3b,60.0,235,689,420,76,300,2.900000e-91,269.0,0.535714,2
41,DRR000185_1467,ref_cb7a7743d531860f,56.2,235,689,419,37,267,1.320000e-86,257.0,0.551313,3
42,DRR000185_1467,ref_78f76ae33a6754f4,55.7,235,689,431,47,277,1.480000e-85,254.0,0.535963,4
43,DRR000185_1467,ref_e56076c6bc1713b1,47.5,238,689,432,12,245,4.310000e-73,223.0,0.541667,5
46,DRR000185_1516,ref_9f8c70fae9ab6aca,31.7,202,917,305,97,291,8.160000e-19,79.0,0.639344,1
48,DRR000185_1800,ref_ae31b49d2c9cff68,30.9,256,1964,260,11,256,1.280000e-31,117.0,0.946154,1
17,DRR000185_238,ref_f4f65a9a0931564b,32.8,259,3128,386,53,298,1.000000e-26,107.0,0.637306,1
18,DRR000185_264,ref_ae31b49d2c9cff68,37.4,257,2563,260,9,259,3.330000e-45,157.0,0.965385,1


## Database construction proof

Build order alternates for three repetitions. The ordered canonical sequence row digest includes identifier, nucleotide sequence, length, GC fraction, and run accession. The build fails if the two variants differ.

In [5]:
display(builds)
display(builds.groupby('variant')[['build_seconds','records_per_second','database_mib']].agg(['median','min','max']))
print(validation)

,variant,repetition,build_seconds,records_per_second,database_mib,page_count,page_size,integrity_check
0,no_bioplastics_metadata,1,1.008090,120352.407992,39.691406,10161,4096,ok
1,bioplastics_metadata,1,1.057776,114699.150159,40.050781,10253,4096,ok
2,bioplastics_metadata,2,1.036486,117055.111177,40.050781,10253,4096,ok
3,no_bioplastics_metadata,2,1.002848,120981.464268,39.691406,10161,4096,ok
4,no_bioplastics_metadata,3,1.016534,119352.581569,39.691406,10161,4096,ok
5,bioplastics_metadata,3,1.076371,112717.662713,40.050781,10253,4096,ok


build_seconds                     records_per_second  \
                               median       min       max             median   
variant                                                                        
bioplastics_metadata         1.057776  1.036486  1.076371      114699.150159   
no_bioplastics_metadata      1.008090  1.002848  1.016534      120352.407992   

                                                      database_mib             \
                                   min            max       median        min   
variant                                                                         
bioplastics_metadata     112717.662713  117055.111177    40.050781  40.050781   
no_bioplastics_metadata  119352.581569  120981.464268    39.691406  39.691406   

                                    
                               max  
variant                             
bioplastics_metadata     40.050781  
no_bioplastics_metadata  39.691406

{'sequence_count': 121326, 'total_bases': 18020607, 'canonical_sequence_rows_sha256': 'de024a79efd1612a03ce82f886315f2c8a004e86b764eb2680c359168360dbab', 'variants_identical': True, 'accessions': ['DRR001152', 'DRR000584', 'DRR001026', 'DRR001104', 'DRR000185', 'DRR001199', 'DRR821254', 'DRR821257', 'DRR821260'], 'plastic_context_accessions': ['DRR821254', 'DRR821257', 'DRR821260'], 'background_control_accessions': ['DRR001152', 'DRR000584', 'DRR001026', 'DRR001104', 'DRR000185', 'DRR001199'], 'reference_sequences': 323, 'reference_annotations': 534, 'candidate_similarity_rows': 28, 'sequences_with_candidate_similarity': 18}


A candidate row is evidence of sequence similarity to a curated plastic-active reference under the stated heuristic thresholds. It is not an inference or experimental proof that the Logan sequence degrades plastic. Plastic-related study context is reported separately from molecular similarity.